# Three-Tier HP Prediction Model

## Model Architecture:
- **Low-CR Model (CR ≤ 1)**: Optimized for very low CR creatures
- **Mid-CR Model (1 < CR ≤ 12)**: Standard model for most creatures
- **High-CR Model (CR > 12)**: Handles legendary/epic creatures

Each model uses the same three-phase approach but with different constraints tuned for their CR range.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Imports successful")

## Load Data

In [ ]:
# Load monster dataset
df = pd.read_csv('../data/dnd5e_monsters_2014.csv')
print(f"📊 Loaded {len(df)} monsters from dataset")

# Load Lazy 5e baseline stats
lazy_5e = pd.read_csv('../data/lazy_5e_monster_stats_by_cr.csv')
print(f"📊 Loaded {len(lazy_5e)} CR baselines from Lazy 5e")

print("\n🔍 Lazy 5e Baseline Stats Preview:")
display(lazy_5e.head(10))

## Parse CR and Create Baseline Lookup Functions

In [ ]:
# Parse CR from string to numeric
def parse_cr(cr_str):
    if pd.isna(cr_str):
        return 0
    cr_str = str(cr_str).strip()
    if '/' in cr_str:
        num, denom = cr_str.split('/')
        return float(num) / float(denom)
    try:
        return float(cr_str)
    except:
        return 0

# Convert Lazy 5e CR to numeric
lazy_5e['cr_numeric'] = lazy_5e['CR'].apply(parse_cr)

# Parse HP from Lazy 5e (extract average from "65 (49-81)" format)
def parse_hp_avg(hp_str):
    if pd.isna(hp_str):
        return 0
    match = re.match(r'(\d+)', str(hp_str))
    return int(match.group(1)) if match else 0

lazy_5e['hp_baseline'] = lazy_5e['HP'].apply(parse_hp_avg)

# Adjust HP baselines: +50% for CR <= 1, +20% for CR >= 2
def adjust_hp_baseline(row):
    if row['cr_numeric'] <= 1.0:
        return row['hp_baseline'] * 1.5
    else:
        return row['hp_baseline'] * 1.2

lazy_5e['hp_baseline'] = lazy_5e.apply(adjust_hp_baseline, axis=1)
print("✅ Adjusted HP baselines: +50% for CR ≤ 1, +20% for CR ≥ 2")

# Parse Attack Bonus - handles both numeric (3) and "+3" format
def parse_bonus(bonus_str):
    if pd.isna(bonus_str):
        return 0
    bonus_str = str(bonus_str).strip()
    # Try to match "+3" format first
    match = re.search(r'\+(\d+)', bonus_str)
    if match:
        return int(match.group(1))
    # Try direct numeric format
    try:
        return int(bonus_str)
    except:
        return 0

lazy_5e['attack_baseline'] = lazy_5e['Attack_Bonus'].apply(parse_bonus)
lazy_5e['ac_baseline'] = lazy_5e['AC_DC']
lazy_5e['dpr_baseline'] = lazy_5e['Damage_Round']

print("✅ Parsed Lazy 5e baselines")
print("\n📊 Baseline Stats by CR:")
display(lazy_5e[['cr_numeric', 'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline']].head(15))

In [ ]:
# Create interpolation functions for baselines
cr_values = lazy_5e['cr_numeric'].values
hp_baseline_interp = interp1d(cr_values, lazy_5e['hp_baseline'].values, 
                               kind='linear', bounds_error=False, fill_value='extrapolate')
ac_baseline_interp = interp1d(cr_values, lazy_5e['ac_baseline'].values,
                               kind='linear', bounds_error=False, fill_value='extrapolate')
attack_baseline_interp = interp1d(cr_values, lazy_5e['attack_baseline'].values,
                                   kind='linear', bounds_error=False, fill_value='extrapolate')
dpr_baseline_interp = interp1d(cr_values, lazy_5e['dpr_baseline'].values,
                                kind='linear', bounds_error=False, fill_value='extrapolate')

def get_baseline_hp(cr):
    return float(hp_baseline_interp(cr))

def get_baseline_ac(cr):
    return float(ac_baseline_interp(cr))

def get_baseline_attack(cr):
    return float(attack_baseline_interp(cr))

def get_baseline_dpr(cr):
    return float(dpr_baseline_interp(cr))

print("✅ Created baseline interpolation functions")

## Feature Engineering (Common to All Models)

In [ ]:
print("⚙️  FEATURE ENGINEERING")
print("="* 60)

# Parse CR
df['cr_numeric'] = df['Challenge_Rating'].apply(parse_cr)

# Parse AC
def parse_ac(ac_str):
    if pd.isna(ac_str):
        return 10
    match = re.search(r'\d+', str(ac_str))
    return int(match.group()) if match else 10

df['ac_value'] = df['AC'].apply(parse_ac)

# Parse speeds
def parse_speed(speed_str, speed_type):
    if pd.isna(speed_str):
        return 0
    speed_str = str(speed_str).lower()
    if speed_type == 'ground':
        match = re.search(r'^(\d+)\s*ft', speed_str)
        return int(match.group(1)) if match else 0
    else:
        pattern = rf'{speed_type}\s+(\d+)\s*ft'
        match = re.search(pattern, speed_str)
        return int(match.group(1)) if match else 0

df['speed_ground'] = df['Speed'].apply(lambda x: parse_speed(x, 'ground'))
df['speed_fly'] = df['Speed'].apply(lambda x: parse_speed(x, 'fly'))
df['speed_swim'] = df['Speed'].apply(lambda x: parse_speed(x, 'swim'))
df['speed_burrow'] = df['Speed'].apply(lambda x: parse_speed(x, 'burrow'))
df['speed_climb'] = df['Speed'].apply(lambda x: parse_speed(x, 'climb'))

df['max_speed'] = df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']].max(axis=1)
df['movement_types_count'] = (df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']] > 0).sum(axis=1)
df['has_flying'] = (df['speed_fly'] > 0).astype(int)

# Parse proficiencies
def count_proficiencies(prof_str):
    if pd.isna(prof_str) or str(prof_str).strip() == '':
        return 0
    return len([x.strip() for x in str(prof_str).split(',') if x.strip()])

df['save_proficiency_count'] = df['Saving_Throws'].apply(count_proficiencies)
df['skill_proficiency_count'] = df['Skills'].apply(count_proficiencies)
df['resistance_count'] = df['Resistances'].apply(count_proficiencies)
df['immunity_count'] = df['Immunities'].apply(count_proficiencies)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_proficiencies)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_proficiencies)

print("✅ Basic features parsed")

In [ ]:
# Parse senses
def has_sense(sense_str, sense_type):
    if pd.isna(sense_str):
        return 0
    return 1 if sense_type in str(sense_str).lower() else 0

def parse_sense_range(sense_str, sense_type):
    if pd.isna(sense_str):
        return 0
    pattern = rf'{sense_type}\s+(\d+)\s*ft'
    match = re.search(pattern, str(sense_str).lower())
    return int(match.group(1)) if match else 0

df['has_darkvision'] = df['Senses'].apply(lambda x: has_sense(x, 'darkvision'))
df['darkvision_range'] = df['Senses'].apply(lambda x: parse_sense_range(x, 'darkvision'))
df['has_blindsight'] = df['Senses'].apply(lambda x: has_sense(x, 'blindsight'))
df['has_truesight'] = df['Senses'].apply(lambda x: has_sense(x, 'truesight'))
df['has_tremorsense'] = df['Senses'].apply(lambda x: has_sense(x, 'tremorsense'))

def parse_passive_perception(sense_str):
    if pd.isna(sense_str):
        return 10
    match = re.search(r'passive\s+perception\s+(\d+)', str(sense_str).lower())
    return int(match.group(1)) if match else 10

df['passive_perception'] = df['Senses'].apply(parse_passive_perception)

print("✅ Senses parsed")

In [ ]:
# Parse abilities from text
def count_abilities(text_str):
    if pd.isna(text_str) or str(text_str).strip() == '':
        return 0
    text = str(text_str).strip()
    entries = [x for x in re.split(r'\n+|\*\s+', text) if x.strip()]
    return len(entries)

df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities)

# Legendary actions
def parse_legendary_actions(leg_str):
    if pd.isna(leg_str) or str(leg_str).strip() == '':
        return 0, 0, 0
    text = str(leg_str).lower()
    has_leg = 1
    count_match = re.search(r'(\d+)\s+legendary\s+actions?', text)
    per_round = int(count_match.group(1)) if count_match else 3
    action_count = len([x for x in re.split(r'\n+|\*\s+', text) if x.strip() and 'can take' not in x.lower()])
    return has_leg, action_count, per_round

df[['has_legendary_actions', 'legendary_action_count', 'legendary_actions_per_round']] = df['Legendary_Actions'].apply(
    lambda x: pd.Series(parse_legendary_actions(x))
)

df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + df['legendary_action_count']

print("✅ Ability counts parsed")

In [ ]:
# Parse specific abilities
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' +
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))

df['has_multiattack'] = combined_abilities.str.contains('multiattack', case=False, na=False).astype(int)

# Combat metrics
def parse_attack_bonus(actions_str):
    if pd.isna(actions_str):
        return 0
    matches = re.findall(r'\+(\d+)\s+to\s+hit', str(actions_str).lower())
    return max([int(m) for m in matches]) if matches else 0

def parse_save_dc(text_str):
    if pd.isna(text_str):
        return 0
    matches = re.findall(r'dc\s+(\d+)', str(text_str).lower())
    return max([int(m) for m in matches]) if matches else 0

df['highest_attack_bonus'] = df['Actions'].apply(parse_attack_bonus)
df['highest_save_dc'] = combined_abilities.apply(parse_save_dc)

# Estimate DPR from action text
def parse_dpr(actions_str):
    if pd.isna(actions_str):
        return 0
    actions_str = str(actions_str).lower()
    total_dpr = 0
    damage_patterns = re.findall(r'(\d+)d(\d+)(?:\s*\+\s*(\d+))?', actions_str)
    for num_dice, die_size, modifier in damage_patterns:
        num_dice = int(num_dice)
        die_size = int(die_size)
        modifier = int(modifier) if modifier else 0
        avg_damage = num_dice * (die_size + 1) / 2 + modifier
        total_dpr += avg_damage
    if 'multiattack' in actions_str:
        total_dpr *= 1.5
    return total_dpr

df['estimated_dpr'] = df['Actions'].apply(parse_dpr)

# Special traits
df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regeneration', case=False, na=False).astype(int)

# Spellcasting
def parse_spellcasting(text_str):
    if pd.isna(text_str):
        return 0, 0
    text = str(text_str).lower()
    if 'spellcasting' not in text and 'innate spellcasting' not in text:
        return 0, 0
    has_spellcasting = 1
    level_match = re.search(r'(\d+)(?:st|nd|rd|th)[-\s]level\s+spellcaster', text)
    if level_match:
        return has_spellcasting, int(level_match.group(1))
    return has_spellcasting, 0

df[['has_spellcasting', 'spellcaster_level']] = df['Traits'].apply(
    lambda x: pd.Series(parse_spellcasting(x))
)

# Size ordinal
size_map = {'Tiny': 0, 'Small': 1, 'Medium': 2, 'Large': 3, 'Huge': 4, 'Gargantuan': 5}
df['size_ordinal'] = df['Size'].map(size_map).fillna(2)

# Grapple
df['has_grapple'] = combined_abilities.str.contains('grapple', case=False, na=False).astype(int)

print("✅ Special abilities parsed")

In [ ]:
# Condition infliction features
conditions = [
    'poisoned', 'blinded', 'charmed', 'deafened', 'frightened',
    'incapacitated', 'paralyzed', 'petrified', 'prone', 'restrained', 'stunned'
]

for condition in conditions:
    feature_name = f'inflicts_{condition}'
    df[feature_name] = combined_abilities.str.contains(condition, case=False, na=False).astype(int)

print(f"✅ {len(conditions)} condition features added")
print("\n✅ Feature engineering complete")

## Calculate Baselines and Deviations

In [ ]:
print("📊 CALCULATING BASELINES AND DEVIATIONS")
print("=" * 60)

# For each monster, get their CR-based baselines
df['hp_baseline'] = df['cr_numeric'].apply(get_baseline_hp)
df['ac_baseline'] = df['cr_numeric'].apply(get_baseline_ac)
df['attack_baseline'] = df['cr_numeric'].apply(get_baseline_attack)
df['dpr_baseline'] = df['cr_numeric'].apply(get_baseline_dpr)

# Calculate deviations from baseline
df['ac_deviation'] = df['ac_value'] - df['ac_baseline']
df['attack_deviation'] = df['highest_attack_bonus'] - df['attack_baseline']
df['dpr_deviation'] = df['estimated_dpr'] - df['dpr_baseline']

# Create scaled features for powerful abilities
print("\n📐 Creating scaled ability features...")
df['has_flying_scaled'] = df['has_flying'] * df['hp_baseline']
df['has_legendary_resistance_scaled'] = df['has_legendary_resistance'] * df['hp_baseline']
df['has_magic_resistance_scaled'] = df['has_magic_resistance'] * df['hp_baseline']
df['has_regeneration_scaled'] = df['has_regeneration'] * df['hp_baseline']
df['has_legendary_actions_scaled'] = df['has_legendary_actions'] * df['hp_baseline']

print("✅ Baselines and deviations calculated")

## Define Feature Sets

In [ ]:
# Phase 1 features: CR baseline
phase1_features = ['hp_baseline']

# Phase 2 features: Deviations and scaled abilities (with fixed penalties)
phase2_features = [
    'ac_deviation', 
    'attack_deviation', 
    'dpr_deviation',
    'has_flying_scaled',
    'has_legendary_resistance_scaled',
    'has_magic_resistance_scaled',
    'has_regeneration_scaled',
    'has_legendary_actions_scaled'
]

# Phase 3 features: Other abilities (learned coefficients)
phase3_features = [
    'speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb',
    'max_speed', 'movement_types_count',
    'save_proficiency_count', 'skill_proficiency_count',
    'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count',
    'has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 'has_tremorsense',
    'passive_perception',
    'trait_count', 'action_count', 'reaction_count', 'bonus_action_count',
    'legendary_action_count', 'legendary_actions_per_round',
    'total_ability_count',
    'has_multiattack', 'highest_save_dc',
    'has_spellcasting', 'spellcaster_level',
    'size_ordinal', 'has_grapple'
]

# Add condition features to Phase 3
for condition in conditions:
    phase3_features.append(f'inflicts_{condition}')

# Combine all features
feature_columns = phase1_features + phase2_features + phase3_features

print(f"📊 Total features: {len(feature_columns)}")
print(f"   - Phase 1 (Baseline): {len(phase1_features)}")
print(f"   - Phase 2 (Deviations + Scaled Abilities): {len(phase2_features)}")
print(f"   - Phase 3 (Other Abilities): {len(phase3_features)}")

## Parse HP and Split Data by CR

In [ ]:
# Parse HP from monster data
def parse_hp(hp_str):
    if pd.isna(hp_str):
        return 0
    hp_str = str(hp_str).strip()
    match = re.match(r'(\d+)', hp_str)
    return int(match.group(1)) if match else 0

df['actual_hp'] = df['HP'].apply(parse_hp)

# Remove rows with HP = 0
df_valid = df[df['actual_hp'] > 0].copy()

print(f"✅ Valid samples: {len(df_valid)} monsters with HP > 0")

# Split data by CR
df_low_cr = df_valid[df_valid['cr_numeric'] <= 1.0].copy()
df_mid_cr = df_valid[(df_valid['cr_numeric'] > 1.0) & (df_valid['cr_numeric'] <= 12.0)].copy()
df_high_cr = df_valid[df_valid['cr_numeric'] > 12.0].copy()

print(f"\n📊 Data Split:")
print(f"   Low-CR (≤ 1):       {len(df_low_cr):3d} monsters")
print(f"   Mid-CR (1-12):      {len(df_mid_cr):3d} monsters")
print(f"   High-CR (> 12):     {len(df_high_cr):3d} monsters")

## Train Low-CR Model (CR ≤ 1)

In [ ]:
print("🤖 TRAINING LOW-CR MODEL (CR ≤ 1)")
print("=" * 80)

# Low-CR specific constraints (lighter penalties for small HP pools)
CONSTRAINTS_LOW = {
    'ac_deviation': -3.0,                      # Reduced from -5
    'attack_deviation': -4.0,                  # Reduced from -6
    'dpr_deviation': -1.5,                     # Reduced from -2.5
    'has_flying_scaled': -0.08,                # Reduced from -0.10
    'has_legendary_resistance_scaled': -0.12,  # Reduced from -0.15
    'has_magic_resistance_scaled': -0.10,      # Reduced from -0.12
    'has_regeneration_scaled': -0.15,          # Reduced from -0.18
    'has_legendary_actions_scaled': -0.06,     # Reduced from -0.08
    # Condition inflictions - set to 0 (no HP impact)
    'inflicts_poisoned': 0.0,
    'inflicts_blinded': 0.0,
    'inflicts_charmed': 0.0,
    'inflicts_deafened': 0.0,
    'inflicts_frightened': 0.0,
    'inflicts_incapacitated': 0.0,
    'inflicts_paralyzed': 0.0,
    'inflicts_petrified': 0.0,
    'inflicts_prone': 0.0,
    'inflicts_restrained': 0.0,
    'inflicts_stunned': 0.0,
}

print("📌 Low-CR Constraints (lighter penalties):")
for feat, val in CONSTRAINTS_LOW.items():
    print(f"   {feat:35s} = {val:+.2f}")

# Prepare data
X_low = df_low_cr[feature_columns].fillna(0)
y_low = df_low_cr['actual_hp']

# Train/test split
X_train_low, X_test_low, y_train_low, y_test_low = train_test_split(
    X_low, y_low, test_size=0.2, random_state=42
)

# Standardize
scaler_low = StandardScaler()
X_train_low_scaled = scaler_low.fit_transform(X_train_low)
X_test_low_scaled = scaler_low.transform(X_test_low)

# Train with constraints
constrained_indices = [feature_columns.index(feat) for feat in CONSTRAINTS_LOW.keys()]
constrained_values_scaled = []

for feat in CONSTRAINTS_LOW.keys():
    idx = feature_columns.index(feat)
    constraint_scaled = CONSTRAINTS_LOW[feat] * scaler_low.scale_[idx]
    constrained_values_scaled.append(constraint_scaled)

y_train_low_adjusted = y_train_low.values.astype(float).copy()
for idx, val in zip(constrained_indices, constrained_values_scaled):
    y_train_low_adjusted -= X_train_low_scaled[:, idx] * val

free_mask = np.ones(len(feature_columns), dtype=bool)
free_mask[constrained_indices] = False
free_indices = np.where(free_mask)[0]

X_train_low_free = X_train_low_scaled[:, free_indices]
XtX = X_train_low_free.T @ X_train_low_free
Xty = X_train_low_free.T @ y_train_low_adjusted
alpha = 1e-6
coef_free = np.linalg.solve(XtX + alpha * np.eye(len(free_indices)), Xty)

coef_low = np.zeros(len(feature_columns))
coef_low[free_indices] = coef_free
for idx, val in zip(constrained_indices, constrained_values_scaled):
    coef_low[idx] = val

intercept_low = y_train_low.mean() - (X_train_low_scaled @ coef_low).mean()

# Evaluate
y_pred_train_low = X_train_low_scaled @ coef_low + intercept_low
y_pred_test_low = X_test_low_scaled @ coef_low + intercept_low

train_r2_low = 1 - np.sum((y_train_low - y_pred_train_low)**2) / np.sum((y_train_low - y_train_low.mean())**2)
test_r2_low = 1 - np.sum((y_test_low - y_pred_test_low)**2) / np.sum((y_test_low - y_test_low.mean())**2)
mae_low = np.mean(np.abs(y_test_low - y_pred_test_low))

print(f"\n📊 Low-CR Model Performance:")
print(f"   Train R²: {train_r2_low:.4f}")
print(f"   Test R²:  {test_r2_low:.4f}")
print(f"   Test MAE: {mae_low:.2f} HP")

## Train Mid-CR Model (1 < CR ≤ 12)

In [ ]:
print("\n🤖 TRAINING MID-CR MODEL (1 < CR ≤ 12)")
print("=" * 80)

# Standard constraints for mid-CR
CONSTRAINTS_MID = {
    'ac_deviation': -5.0,
    'attack_deviation': -6.0,
    'dpr_deviation': -2.5,
    'has_flying_scaled': -0.10,
    'has_legendary_resistance_scaled': -0.15,
    'has_magic_resistance_scaled': -0.12,
    'has_regeneration_scaled': -0.18,
    'has_legendary_actions_scaled': -0.08,
    # Condition inflictions - set to 0 (no HP impact)
    'inflicts_poisoned': 0.0,
    'inflicts_blinded': 0.0,
    'inflicts_charmed': 0.0,
    'inflicts_deafened': 0.0,
    'inflicts_frightened': 0.0,
    'inflicts_incapacitated': 0.0,
    'inflicts_paralyzed': 0.0,
    'inflicts_petrified': 0.0,
    'inflicts_prone': 0.0,
    'inflicts_restrained': 0.0,
    'inflicts_stunned': 0.0,
}

print("📌 Mid-CR Constraints (standard):")
for feat, val in CONSTRAINTS_MID.items():
    print(f"   {feat:35s} = {val:+.2f}")

# Prepare data
X_mid = df_mid_cr[feature_columns].fillna(0)
y_mid = df_mid_cr['actual_hp']

# Train/test split
X_train_mid, X_test_mid, y_train_mid, y_test_mid = train_test_split(
    X_mid, y_mid, test_size=0.2, random_state=42
)

# Standardize
scaler_mid = StandardScaler()
X_train_mid_scaled = scaler_mid.fit_transform(X_train_mid)
X_test_mid_scaled = scaler_mid.transform(X_test_mid)

# Train with constraints (same logic as low-CR)
constrained_indices = [feature_columns.index(feat) for feat in CONSTRAINTS_MID.keys()]
constrained_values_scaled = []

for feat in CONSTRAINTS_MID.keys():
    idx = feature_columns.index(feat)
    constraint_scaled = CONSTRAINTS_MID[feat] * scaler_mid.scale_[idx]
    constrained_values_scaled.append(constraint_scaled)

y_train_mid_adjusted = y_train_mid.values.astype(float).copy()
for idx, val in zip(constrained_indices, constrained_values_scaled):
    y_train_mid_adjusted -= X_train_mid_scaled[:, idx] * val

free_mask = np.ones(len(feature_columns), dtype=bool)
free_mask[constrained_indices] = False
free_indices = np.where(free_mask)[0]

X_train_mid_free = X_train_mid_scaled[:, free_indices]
XtX = X_train_mid_free.T @ X_train_mid_free
Xty = X_train_mid_free.T @ y_train_mid_adjusted
coef_free = np.linalg.solve(XtX + alpha * np.eye(len(free_indices)), Xty)

coef_mid = np.zeros(len(feature_columns))
coef_mid[free_indices] = coef_free
for idx, val in zip(constrained_indices, constrained_values_scaled):
    coef_mid[idx] = val

intercept_mid = y_train_mid.mean() - (X_train_mid_scaled @ coef_mid).mean()

# Evaluate
y_pred_train_mid = X_train_mid_scaled @ coef_mid + intercept_mid
y_pred_test_mid = X_test_mid_scaled @ coef_mid + intercept_mid

train_r2_mid = 1 - np.sum((y_train_mid - y_pred_train_mid)**2) / np.sum((y_train_mid - y_train_mid.mean())**2)
test_r2_mid = 1 - np.sum((y_test_mid - y_pred_test_mid)**2) / np.sum((y_test_mid - y_test_mid.mean())**2)
mae_mid = np.mean(np.abs(y_test_mid - y_pred_test_mid))

print(f"\n📊 Mid-CR Model Performance:")
print(f"   Train R²: {train_r2_mid:.4f}")
print(f"   Test R²:  {test_r2_mid:.4f}")
print(f"   Test MAE: {mae_mid:.2f} HP")

## Train High-CR Model (CR > 12)

In [ ]:
print("\n🤖 TRAINING HIGH-CR MODEL (CR > 12)")
print("=" * 80)

# High-CR constraints (heavier penalties for large HP pools)
CONSTRAINTS_HIGH = {
    'ac_deviation': -7.0,                      # Increased from -5
    'attack_deviation': -8.0,                  # Increased from -6
    'dpr_deviation': -3.5,                     # Increased from -2.5
    'has_flying_scaled': -0.12,                # Increased from -0.10
    'has_legendary_resistance_scaled': -0.18,  # Increased from -0.15
    'has_magic_resistance_scaled': -0.15,      # Increased from -0.12
    'has_regeneration_scaled': -0.22,          # Increased from -0.18
    'has_legendary_actions_scaled': -0.10,     # Increased from -0.08
    # Condition inflictions - set to 0 (no HP impact)
    'inflicts_poisoned': 0.0,
    'inflicts_blinded': 0.0,
    'inflicts_charmed': 0.0,
    'inflicts_deafened': 0.0,
    'inflicts_frightened': 0.0,
    'inflicts_incapacitated': 0.0,
    'inflicts_paralyzed': 0.0,
    'inflicts_petrified': 0.0,
    'inflicts_prone': 0.0,
    'inflicts_restrained': 0.0,
    'inflicts_stunned': 0.0,
}

print("📌 High-CR Constraints (heavier penalties):")
for feat, val in CONSTRAINTS_HIGH.items():
    print(f"   {feat:35s} = {val:+.2f}")

# Prepare data
X_high = df_high_cr[feature_columns].fillna(0)
y_high = df_high_cr['actual_hp']

# Train/test split
X_train_high, X_test_high, y_train_high, y_test_high = train_test_split(
    X_high, y_high, test_size=0.2, random_state=42
)

# Standardize
scaler_high = StandardScaler()
X_train_high_scaled = scaler_high.fit_transform(X_train_high)
X_test_high_scaled = scaler_high.transform(X_test_high)

# Train with constraints (same logic)
constrained_indices = [feature_columns.index(feat) for feat in CONSTRAINTS_HIGH.keys()]
constrained_values_scaled = []

for feat in CONSTRAINTS_HIGH.keys():
    idx = feature_columns.index(feat)
    constraint_scaled = CONSTRAINTS_HIGH[feat] * scaler_high.scale_[idx]
    constrained_values_scaled.append(constraint_scaled)

y_train_high_adjusted = y_train_high.values.astype(float).copy()
for idx, val in zip(constrained_indices, constrained_values_scaled):
    y_train_high_adjusted -= X_train_high_scaled[:, idx] * val

free_mask = np.ones(len(feature_columns), dtype=bool)
free_mask[constrained_indices] = False
free_indices = np.where(free_mask)[0]

X_train_high_free = X_train_high_scaled[:, free_indices]
XtX = X_train_high_free.T @ X_train_high_free
Xty = X_train_high_free.T @ y_train_high_adjusted
coef_free = np.linalg.solve(XtX + alpha * np.eye(len(free_indices)), Xty)

coef_high = np.zeros(len(feature_columns))
coef_high[free_indices] = coef_free
for idx, val in zip(constrained_indices, constrained_values_scaled):
    coef_high[idx] = val

intercept_high = y_train_high.mean() - (X_train_high_scaled @ coef_high).mean()

# Evaluate
y_pred_train_high = X_train_high_scaled @ coef_high + intercept_high
y_pred_test_high = X_test_high_scaled @ coef_high + intercept_high

train_r2_high = 1 - np.sum((y_train_high - y_pred_train_high)**2) / np.sum((y_train_high - y_train_high.mean())**2)
test_r2_high = 1 - np.sum((y_test_high - y_pred_test_high)**2) / np.sum((y_test_high - y_test_high.mean())**2)
mae_high = np.mean(np.abs(y_test_high - y_pred_test_high))

print(f"\n📊 High-CR Model Performance:")
print(f"   Train R²: {train_r2_high:.4f}")
print(f"   Test R²:  {test_r2_high:.4f}")
print(f"   Test MAE: {mae_high:.2f} HP")

## Save All Models

In [ ]:
print("\n💾 SAVING ALL MODELS")
print("=" * 80)

# Save Low-CR Model
model_low = {
    'coef': coef_low,
    'intercept': intercept_low,
    'scaler': scaler_low,
    'feature_columns': feature_columns,
    'constraints': CONSTRAINTS_LOW,
    'cr_range': (0, 1.0),
    'test_r2': test_r2_low,
    'test_mae': mae_low
}

with open('../pickled_models/hp_model_low_cr.pkl', 'wb') as f:
    pickle.dump(model_low, f)
print("✅ Saved: ../pickled_models/hp_model_low_cr.pkl")

# Save Mid-CR Model
model_mid = {
    'coef': coef_mid,
    'intercept': intercept_mid,
    'scaler': scaler_mid,
    'feature_columns': feature_columns,
    'constraints': CONSTRAINTS_MID,
    'cr_range': (1.0, 12.0),
    'test_r2': test_r2_mid,
    'test_mae': mae_mid
}

with open('../pickled_models/hp_model_mid_cr.pkl', 'wb') as f:
    pickle.dump(model_mid, f)
print("✅ Saved: ../pickled_models/hp_model_mid_cr.pkl")

# Save High-CR Model
model_high = {
    'coef': coef_high,
    'intercept': intercept_high,
    'scaler': scaler_high,
    'feature_columns': feature_columns,
    'constraints': CONSTRAINTS_HIGH,
    'cr_range': (12.0, 30.0),
    'test_r2': test_r2_high,
    'test_mae': mae_high
}

with open('../pickled_models/hp_model_high_cr.pkl', 'wb') as f:
    pickle.dump(model_high, f)
print("✅ Saved: ../pickled_models/hp_model_high_cr.pkl")

# Save baseline lookup data
baseline_data = {
    'cr_values': lazy_5e['cr_numeric'].tolist(),
    'hp_baseline': lazy_5e['hp_baseline'].tolist(),
    'ac_baseline': lazy_5e['ac_baseline'].tolist(),
    'attack_baseline': lazy_5e['attack_baseline'].tolist(),
    'dpr_baseline': lazy_5e['dpr_baseline'].tolist()
}

with open('../data/baseline_lookup_three_tier.json', 'w') as f:
    json.dump(baseline_data, f, indent=2)
print("✅ Saved: ../data/baseline_lookup_three_tier.json")

## Export for Web App

In [ ]:
print("\n📦 EXPORTING FOR WEB APP")
print("=" * 80)

# Helper function to unscale coefficients
def unscale_coefficients(coef, scaler, feature_columns):
    return {feat: float(coef[i] / scaler.scale_[i]) 
            for i, feat in enumerate(feature_columns)}

# Export all three models
web_export = {
    'model_type': 'three_tier_hp_model',
    'description': 'Three separate models optimized for different CR ranges',
    'baseline_data': baseline_data,
    'feature_columns': feature_columns,
    'phase1_features': phase1_features,
    'phase2_features': phase2_features,
    'phase3_features': phase3_features,
    
    'models': {
        'low_cr': {
            'cr_range': [0, 1.0],
            'intercept': float(intercept_low),
            'coefficients': unscale_coefficients(coef_low, scaler_low, feature_columns),
            'scaler_mean': {feat: float(m) for feat, m in zip(feature_columns, scaler_low.mean_)},
            'scaler_scale': {feat: float(s) for feat, s in zip(feature_columns, scaler_low.scale_)},
            'constraints': CONSTRAINTS_LOW,
            'test_r2': float(test_r2_low),
            'test_mae': float(mae_low)
        },
        'mid_cr': {
            'cr_range': [1.0, 12.0],
            'intercept': float(intercept_mid),
            'coefficients': unscale_coefficients(coef_mid, scaler_mid, feature_columns),
            'scaler_mean': {feat: float(m) for feat, m in zip(feature_columns, scaler_mid.mean_)},
            'scaler_scale': {feat: float(s) for feat, s in zip(feature_columns, scaler_mid.scale_)},
            'constraints': CONSTRAINTS_MID,
            'test_r2': float(test_r2_mid),
            'test_mae': float(mae_mid)
        },
        'high_cr': {
            'cr_range': [12.0, 30.0],
            'intercept': float(intercept_high),
            'coefficients': unscale_coefficients(coef_high, scaler_high, feature_columns),
            'scaler_mean': {feat: float(m) for feat, m in zip(feature_columns, scaler_high.mean_)},
            'scaler_scale': {feat: float(s) for feat, s in zip(feature_columns, scaler_high.scale_)},
            'constraints': CONSTRAINTS_HIGH,
            'test_r2': float(test_r2_high),
            'test_mae': float(mae_high)
        }
    }
}

with open('../monster-builder-v2/model_data.json', 'w') as f:
    json.dump(web_export, f, indent=2)
print("✅ Saved: ../monster-builder-v2/model_data.json")

print("\n✅ All models exported successfully!")

## Summary

In [ ]:
print("\n" + "=" * 80)
print("🎉 THREE-TIER HP MODEL TRAINING COMPLETE")
print("=" * 80)

print("\n📊 MODEL PERFORMANCE SUMMARY:")
print(f"\n   Low-CR Model (CR ≤ 1):")
print(f"      Training samples: {len(y_train_low)}")
print(f"      Test R²:  {test_r2_low:.4f}")
print(f"      Test MAE: {mae_low:.2f} HP")

print(f"\n   Mid-CR Model (1 < CR ≤ 12):")
print(f"      Training samples: {len(y_train_mid)}")
print(f"      Test R²:  {test_r2_mid:.4f}")
print(f"      Test MAE: {mae_mid:.2f} HP")

print(f"\n   High-CR Model (CR > 12):")
print(f"      Training samples: {len(y_train_high)}")
print(f"      Test R²:  {test_r2_high:.4f}")
print(f"      Test MAE: {mae_high:.2f} HP")

print("\n📁 Files Created:")
print("   ✅ pickled_models/hp_model_low_cr.pkl")
print("   ✅ pickled_models/hp_model_mid_cr.pkl")
print("   ✅ pickled_models/hp_model_high_cr.pkl")
print("   ✅ data/baseline_lookup_three_tier.json")
print("   ✅ monster-builder-v2/model_data.json (web app export)")

print("\n✨ Ready for integration into monster-builder-v2!")